
# Phishing Dataset Builder (Simple & Clean)

This notebook builds a **50:50 phishing vs legitimate URL dataset** using:

- URLHaus (FULL ZIP CSV)
- OpenPhish
- PhishTank
- Majestic Million

Includes:
- Login portal augmentation
- Deep-path legitimate URLs
- Cleaning + Exact balancing


In [ ]:

import pandas as pd
import requests
import zipfile
from io import BytesIO

TARGET_TOTAL = 200000

NUM_PHISH = TARGET_TOTAL // 2
NUM_LEGIT = TARGET_TOTAL // 2

OUTPUT_FILE = "dataset_new.csv"

print("Dataset Builder Started")


In [ ]:

def download_urlhaus():

    print("Downloading URLHaus FULL ZIP...")

    r = requests.get(
        "https://urlhaus.abuse.ch/downloads/csv/",
        headers={"User-Agent":"Mozilla/5.0"},
        timeout=60
    )

    z = zipfile.ZipFile(BytesIO(r.content))

    csv_file = [f for f in z.namelist() if f.endswith(".csv")][0]

    with z.open(csv_file) as f:

        df = pd.read_csv(
            f,
            comment="#",
            header=None
        )

    df = df[[2]]
    df.columns=["url"]

    return df


In [ ]:

def download_openphish():

    print("Downloading OpenPhish...")

    r = requests.get("https://openphish.com/feed.txt")

    return pd.DataFrame(
        r.text.splitlines(),
        columns=["url"]
    )


def download_phishtank():

    print("Downloading PhishTank...")

    df = pd.read_csv(
        "https://data.phishtank.com/data/online-valid.csv"
    )

    return df[["url"]]


In [ ]:

def download_majestic():

    print("Downloading Majestic Million...")

    df = pd.read_csv(
        "https://downloads.majestic.com/majestic_million.csv"
    )

    urls = ["https://" + d for d in df["Domain"].dropna()]

    return pd.DataFrame(urls,columns=["url"])


In [ ]:

df_phish_pool = pd.concat([

    download_urlhaus(),
    download_openphish(),
    download_phishtank()

])

df_phish_pool.drop_duplicates(inplace=True)

print("Phishing Pool:",len(df_phish_pool))

NUM_PHISH = min(NUM_PHISH,len(df_phish_pool))

df_phish = df_phish_pool.sample(NUM_PHISH,random_state=42)

df_phish["label"] = 1

df_phish.head()


In [ ]:

df_legit_pool = download_majestic()

df_legit = df_legit_pool.sample(NUM_PHISH,random_state=42)

df_legit["label"] = 0

df_legit.head()


In [ ]:

extra_legit = [

"https://accounts.google.com",
"https://login.microsoftonline.com",
"https://github.com/login",
"https://paypal.com/signin"

]

df_legit = pd.concat([

    df_legit,
    pd.DataFrame({"url":extra_legit,"label":0})

])

print("Login Augmentation Added")


In [ ]:

paths = ["/docs","/security","/help"]

sample_domains = df_legit_pool.sample(5000,random_state=42)["url"]

deep_urls = []

for d in sample_domains:

    d = d.rstrip("/")

    for p in paths:

        deep_urls.append(d+p)

df_legit = pd.concat([

    df_legit,
    pd.DataFrame({"url":deep_urls,"label":0})

])

print("Deep Paths Added")


In [ ]:

df_final = pd.concat([df_legit,df_phish])

df_final.drop_duplicates(subset="url",inplace=True)

df_final = df_final[
    df_final.url.str.startswith("http")
]

print("After Cleaning:",df_final.shape)


In [ ]:

ph = df_final[df_final.label==1]

lg = df_final[df_final.label==0]

n = min(len(ph),len(lg))

df_final = pd.concat([

    ph.sample(n,random_state=42),
    lg.sample(n,random_state=42)

])

df_final = df_final.sample(frac=1,random_state=42)

print(df_final.label.value_counts())

print("Final Size:",df_final.shape)


In [ ]:

df_final.to_csv(OUTPUT_FILE,index=False)

print("Saved:",OUTPUT_FILE)
